# UnitRoot-Tools 0.1.0rc1 — Google Colab platform validation

**Prepared, not executed in Colab. No Colab pass is claimed.** Upload the
candidate wheel, source archive and SHA256SUMS from the instructor-reviewed
artifact folder. This notebook uses an isolated virtual environment to avoid
changing Colab's preinstalled packages. It runs all tests, saved numerical
audits and teaching scripts via that environment's Python; it does not claim
compatibility with arbitrary preloaded Colab kernel dependencies.

Run every cell in order. Preserve the downloaded results even if a check fails.
A green notebook cell alone is insufficient: inspect status.json and numerical
reports. No GitHub/PyPI publication, remote credentials or R installation is
needed. Dependency installation needs network access. Do not use Python -O.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, subprocess, sys, tarfile, tempfile

uploaded = files.upload()  # select wheel, .tar.gz and SHA256SUMS
workspace = Path(tempfile.mkdtemp(prefix="unitroot-rc-"))
wheel_name = "unitroot_tools-0.1.0rc1-py3-none-any.whl"
source_name = "unitroot_tools-0.1.0rc1.tar.gz"
assert {wheel_name, source_name, "SHA256SUMS"} <= set(uploaded)
for name in (wheel_name, source_name, "SHA256SUMS"):
    (workspace / name).write_bytes(uploaded[name])
checksums = {}
for line in uploaded["SHA256SUMS"].decode().splitlines():
    digest, name = line.split(maxsplit=1)
    checksums[name.lstrip("*")] = digest
for name in (wheel_name, source_name):
    assert hashlib.sha256(uploaded[name]).hexdigest() == checksums[name], name
print("Candidate checksums verified", workspace)


In [ ]:
with tarfile.open(workspace / source_name) as archive:
    archive.extractall(workspace / "source", filter="data")
source = workspace / "source" / "unitroot_tools-0.1.0rc1"
assert (source / "validation/run_platform_validation.py").is_file()
subprocess.run([sys.executable, "-m", "venv", str(workspace / "env")], check=True)
python = workspace / "env/bin/python"
subprocess.run([
    str(python), "-m", "pip", "install", "--only-binary=:all:",
    str(workspace / wheel_name) + "[test]", "arch==8.0.0",
    "numpy==2.4.4", "scipy==1.17.1", "statsmodels==0.15.0", "pandas==3.1.0",
    "pytest==9.1.1", "pytest-cov==7.1.0",
], check=True)


In [ ]:
result = subprocess.run([
    str(python), str(source / "validation/run_platform_validation.py"),
    "--source", str(source), "--out", str(workspace / "results"),
])
status_path = workspace / "results/status.json"
assert status_path.exists(), "Validation did not finish; inspect the preceding error."
status = json.loads(status_path.read_text())
print(json.dumps(status, indent=2))
print((workspace / "results/installed-smoke.log").read_text())


## Inspect teaching output and retain the evidence

The full runner already executes all three teaching examples. This cell shows
the walkthrough output without importing the package into Colab's base kernel.
Null hypotheses and rejection tails remain distinct; failure to reject does
not prove stationarity or a unit root. The results archive includes versions,
BLAS configuration, test XML, fixed tolerances and numerical differences.


In [ ]:
import shutil
print((workspace / "results/teaching_walkthrough.py.log").read_text())
archive_path = shutil.make_archive(str(workspace / "platform-results"), "zip", workspace / "results")
files.download(archive_path)
assert result.returncode == 0 and status["all_passed"], "Some checks failed; retain and investigate the results."
